### Практическое задание
В этот раз будем решать задачу прогнозирования оттока продавцов в Красноярском крае.

Бизнес считает, что клиент утек, если в начале месяца у него были живые объявления, а в последний день месяца наблюдаем следующую картину: живых объявлений нет, а последнее было удалено из сервиса более 2 недель назад.

Особенностью этого задания является использование нестандартной метрики качества. Это потребует написания ощутимого количества кастомного кода, но поможет нам лучше разобраться, как работают логистическая регрессия и grid search.

Это большое и насыщенное задание. Выделите под него достаточно времени.

### Задача 1. Выгрузка и подготовка данных (1/2)
Представим, что сейчас 1 октября 2022 года. У нашего сервиса есть пользователи с активными на данный момент объявлениями. Мы хотим предсказать, что случится с этими пользователями через месяц, т.е. на момент 1 ноября 2022 года.

Тут может быть три варианта:

1. На момент 1 ноября у пользователя всё ещё есть активные объявления (эти же или новые). Оттока не произошло, человек продолжает пользоваться нашим сервисом.
2. На момент 1 ноября у пользователя нет активных объявлений, причём нет давно (более 2 недель). Это мы считаем оттоком. Пользователь перестал активно пользоваться нашим сервисом.
3. На момент 1 ноября у пользователя нет активных объявлений. При этом недавно (в пределах 2 недель) объявления ещё были. Тут пока не понятно, отток это или нет. Таких пользователей мы просто отбросим и сконцентрируемся на том, чтобы научиться различать первые две группы (явный "не отток" и явный отток).

Какие данные мы можем использовать? Прежде всего, пользователей характеризуют их объявления. Плюс, есть какая-то общая информация о пользователях. Могут быть ещё какие-то варианты, помогающие описать поведение пользователей (например, сделанные ими транзакции), но к ним мы вернёмся чуть позже.

Также обратите внимание на даты, за которые нам нужны объявления. Для подготовки признаков нам нужны объявления, активные на 1 октября 2022 года. Дальше мы не можем заглядывать: дальше — "будущее", оно ещё не наступило. С другой стороны, для подготовки таргета нам нужно как раз заглянуть в будущее и посмотреть, что будет происходить с объявлениями с 1 октября по 1 ноября.

Поэтому мы изначально вытащим данные по объявлениям за весь месяц и по ним посчитаем таргет. А потом оставим только данные за 1 октября и по этим данным уже будем считать фичи.

#### Выгрузка данных
Напишем SQL-запрос для получения данных из Кликхауза.

Объявления лежат в таблице `live_adverts`. Возьмём все имеющиеся в таблице столбцы.

Сразу присоединим общую информацию о пользователе. В нашем случае это будет только одно поле `user_type_cars_name` из таблицы `user_passports`. Используйте left join по `passport_id`.

Отфильтруем по времени и месту. Оставим только строки, в которых:

- `execution_date` находится между `2022-10-01` и `2022-11-01` включительно: берём объявления только за рассматриваемый период (как обсуждали выше, весь месяц нужен нам для таргета, а для фич потом оставим только `2022-10-01`).
- `region` это 'Красноярск': проводим эксперимент в этом регионе.

После выполнения запроса сразу вызовите `pd.to_datetime` для `execution_date` и `created_at`.

Давайте сверимся. Мы выгрузили датасет с сырыми данными. Сколько в нём строк?

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

from sklearn.compose import make_column_selector as selector
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.pipeline import Pipeline 
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, make_scorer, precision_score, precision_recall_curve, auc, average_precision_score
from sklearn.inspection import permutation_importance
from clickhouse_driver import Client
from config import user_id, password

import warnings
warnings.filterwarnings('ignore')

# работаем с двумя БД, поэтому создаем два подключения.
readDB = 'hardda'
readWriteDB = 'hardda_student_data'


# Создаем соединение с ClickHouse
client = Client(
    host='clickhouse.lab.karpov.courses',
    port=9000,
    user=user_id,
    password=password,
    database=readDB
)

def get_data(query):
    """
    Вытягивает данные из clickhouse в виде Dataframe
    
    query - запрос
    """
    result, columns = client.execute(query, with_column_types=True)
    return pd.DataFrame(result, columns=[tuple[0] for tuple in columns])

In [2]:
query = '''
SELECT la.*,
    up.user_type_cars_name
FROM live_adverts la
LEFT JOIN user_passports up 
ON la.passport_id = up.passport_id
WHERE execution_date BETWEEN '2022-10-01' AND '2022-11-01'
    AND region = 'Красноярск'
'''

df = get_data(query)
df['execution_date'] = pd.to_datetime(df['execution_date'], format='%Y-%m-%d')
df['created_at'] = pd.to_datetime(df['created_at'], format='%Y-%m-%d')
df.shape[0]

195131

In [3]:
df.head()

,execution_date,advert_id,created_at,price,region,user_id,platform,auto_brand,auto_model,passport_id,year,userType,user_type_cars_name
0,2022-10-31,139593230,2014-05-14 08:00:22,0,Красноярск,123488190,desktop,Unknown,Unknown model,123508931,0,0,cars_simple
1,2022-10-31,142255830,2014-10-21 17:43:15,12000000,Красноярск,123510982,unknown,Unknown,Unknown model,123531907,2012,0,cars_simple
2,2022-10-31,143662578,2015-01-09 16:47:38,12000000,Красноярск,123510982,unknown,Unknown,Unknown model,123531907,2014,0,cars_simple
3,2022-10-31,144746434,2015-02-28 23:20:10,2400000,Красноярск,123464603,desktop,Mercedes-Benz,S 350,123476026,2008,0,cars_simple
4,2022-10-31,147966866,2015-09-07 19:48:43,2000000,Красноярск,123799467,desktop,Mercedes-Benz,Vito,123820425,2011,0,cars_simple


### Задача 1. Выгрузка и подготовка данных (2/2)
У нас есть датасет с сырыми данными. На этом шаге мы соберём из него датасет, в котором будут только интересующие нас объявления: актуальные на 1 октября и только такие, где мы знаем, утечёт создатель этого объявления через месяц или нет.

#### Считаем таргет
Подумайте, как бы вы посчитали таргет для каждого `passport_id`, учитывая описанное на предыдущем степе. Ниже один из вариантов, как это можно сделать.

Для каждого пользователя (`passport_id`) посчитайте минимальную и максимальную дату активного объявления (`execution_date`). Проще всего сразу сделать датафрейм, где будет `passport_id` и две даты, т.к. потом его будет удобно фильтровать, добавлять столбцы и т.д.

Оставьте только строки, где минимальная дата это 1 октября. У остальных пользователей не было объявлений на дату начала эксперимента, и нас они не интересуют.

Посчитайте разницу в днях между 1 ноября 2022 г. и максимальной датой объявления пользователя. Получится количество дней до 1 ноября, в течение которых пользователь был неактивен. Добавьте такой столбец в датафрейм.

Оставьте только строки, где полученное количество неактивных дней перед 1 ноября либо равно 0 (тогда это очевидный "не отток"), либо строго больше 14 (это считаем оттоком). Теперь у нас остались только пользователи, про которых мы чётко можем сказать, отток это или нет, а всех остальных, где "неясно", мы отбросили.

Выставим таргет. Добавьте столбец `churn`: если количество неактивных дней более 14, его значение равно 1 (отток), иначе 0 (не отток).

Оставьте только столбцы `passport_id` и `churn` (остальные были вспомогательными для расчётов).

#### Собираем датасет с объявлениями
Возвращаемся к датафрейму, полученному на предыдущем степе, со всеми объявлениями за месяц.

Возьмите из него только строки, в которых `execution_date` равна `2022-10-01` (таргет мы уже собрали, а для признаков нам нужны только объявления, актуальные на дату начала эксперимента).

Сджойните результат с полученным до этого датафреймом (содержащим таргет).

#### Что дальше?
Теперь у нас есть датафрейм с объявлениями, актуальными на 1 октября, и в каждом есть информация, продолжит ли пользоваться сервисом создатель этого объявления через месяц.

Теперь нужно каким-то образом сгруппировать информацию, чтобы из датасета с объявлениями перейти к датасету с пользователями, в котором будут признаки пользователя и таргет.

Прежде чем идти дальше, подумайте, как это сделали бы вы.

И давайте снова убедимся, что у нас получились одинаковые результаты.

Теперь у нас есть датасет с объявлениями и таргетом. Через запятую и пробел введите количество строк в полученном датафрейме и количество уникальных пользователей в нём. Например, `5432`, `4321`.

In [4]:
min_max_date_df = df.groupby('passport_id', as_index=False).agg(min_date=('execution_date', 'min'), max_date=('execution_date', 'max'))
min_max_date_df.head()

,passport_id,min_date,max_date
0,123463370,2022-10-02,2022-10-31
1,123464228,2022-10-01,2022-10-26
2,123467852,2022-10-01,2022-11-01
3,123468245,2022-10-28,2022-11-01
4,123468633,2022-10-25,2022-10-31


In [5]:
min_max_date_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13293 entries, 0 to 13292
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   passport_id  13293 non-null  int64         
 1   min_date     13293 non-null  datetime64[ns]
 2   max_date     13293 non-null  datetime64[ns]
dtypes: datetime64[ns](2), int64(1)
memory usage: 311.7 KB


In [6]:
date_sorted_df = min_max_date_df[min_max_date_df['min_date'] == '2022-10-01']

end_date = datetime(2022, 11, 1)

date_sorted_df['inactive_days'] = (end_date - date_sorted_df['max_date']).dt.days

filtered_df = date_sorted_df[
    (date_sorted_df['inactive_days'] == 0) | 
    (date_sorted_df['inactive_days'] > 14)
]

filtered_df['churn'] = filtered_df['inactive_days'].apply(
    lambda x: 1 if x > 14 else 0
)

final_df = filtered_df[['passport_id', 'churn']]

final_df.head()

,passport_id,churn
2,123467852,0
5,123469393,0
7,123469843,0
8,123475067,0
10,123476026,0


In [7]:
start_date_ads_df = df[df['execution_date'] == '2022-10-01']

merged_df = start_date_ads_df.merge(final_df, on='passport_id', how='inner')

In [8]:
print(f"Ответ: {len(merged_df)}, {merged_df['passport_id'].nunique()}")

Ответ: 4367, 3441


### Задача 2. Признаки пользователя (1/5)
Нам нужно сгруппировать и сагрегировать объявления по каждому пользователю, чтобы из датафрейма с объявлениями получить датафрейм с пользователями.

Будем решать задачу итеративно, постепенно добавляя в агрегацию новые поля. Иногда нам нужно будет сделать что-то дополнительно перед группировкой и агрегацией.

#### opening_adverts_amount
Начнём с самого очевидного. По каждому пользователю посчитаем текущее количество живых объявлений. Назовите получившийся столбец `opening_adverts_amount`.

Идея признака проста: если у пользователя много активных объявлений, менее вероятно, что он утечёт в ближайший месяц.

#### price
Посчитаем среднюю цену объявлений пользователя.

Это менее очевидный признак, но раз в объявлениях есть цена, нужно это использовать. Например, может оказаться, что более дорогие машины продаются дольше, и пользователь с меньшей вероятностью уйдёт в ближайший месяц.

Со столбцом `price` есть небольшая проблема, которую нам нужно решить перед группировкой: есть объявления с ценой равной 0. Вряд ли автомобили отдают бесплатно, более вероятно, что это пропущенные значения.

Перед группировкой замените значение 0 на `np.nan` в столбце с ценами. И уже после этого добавьте столбец в агрегацию, посчитав среднее. Полученный столбец назовите `price`.

При агрегации `pandas` посчитает среднее по непустым ценам. Но где-то останутся пропуски. С этим мы будем разбираться потом, в пайплайне предподготовки данных.

В поле ниже введите два числа через запятую и пробел:

- Среднее значение `opening_adverts_amount` в полученном датафрейме, округлённое до 3 знаков.
- Максимальное значение `price`, округлённое до целого.

Например, `1.234, 56789000`.

In [9]:
merged_df['price'] = merged_df['price'].replace({0: np.nan})

In [10]:
feature_df = merged_df.groupby('passport_id', as_index=False).agg(
    opening_adverts_amount=('advert_id', 'count'),
    price=('price', 'mean'),
)

In [11]:
print(f"Ответ: {round(feature_df['opening_adverts_amount'].mean(), 3)}, {round(feature_df['price'].max())}")

Ответ: 1.269, 24000000


### Задача 2. Признаки пользователя (2/5)
Добавим ещё два признака.

#### auto_age
Посчитаем средний возраст автомобилей пользователя. Перед агрегацией нужно сначала сделать несколько вещей:

1. Исправьте тип столбца year на `int`.
2. Как и в случае с ценой, встречаются значения года, равные 0. Если про цену ещё могли быть сомнения, то тут это точно пропуск. Замените значение 0 на `np.nan`.
3. Добавьте в датасет с объявлениями столбец `auto_age`, полученный как разность между 2022 ("текущим" годом) и year.

Теперь у нас в каждом объявлении есть возраст автомобиля. Добавьте его в агрегацию, взяв среднее. Полученный столбец назовите тоже `auto_age`.

#### advert_age
Также посчитаем средний возраст объявлений пользователя (в днях).

Логика признака может быть разной. С одной стороны, если объявление давно висит, возможно, оно уже собрало достаточно просмотров. Машина скоро продастся, а клиент, если он — физическое лицо, после продажи может навсегда пропасть с платформы. С другой стороны, если у пользователя есть активные объявления, которые давно висят, очень может быть, что они повисят и ещё месяц. В любом  случае, есть вероятность, что такой признак даст модели полезную информацию.

Сначала для датасета с объявлениями посчитайте разность между датой `2022-10-01` и датой `created_at`. Обратите внимание, что из `created_at` берём только дату. Время можно отбросить, вызвав `.dt.normalize()`. Из полученной разности возьмите количество дней и добавьте в датафрейм как столбец `advert_age`.

При агрегации возьмите среднее по этому столбцу. Полученный столбец назовите тоже `advert_age`.

--- 

В поле ниже через запятую и пробел введите два числа:

- Медиану `auto_age`, округлённую до целого.
- Максимальный `advert_age`, тоже округлённый до целого.

Например, 12, 3456.

In [12]:
merged_df['year'] = merged_df['year'].astype('Int64')
merged_df['year'] = merged_df['year'].replace(0, np.nan)

merged_df['auto_age'] = 2022 - merged_df['year']
merged_df['created_at'] = pd.to_datetime(merged_df['created_at'])
start_experiment = datetime(2022, 10, 1)
merged_df['advert_age'] = (start_experiment - merged_df['created_at'].dt.normalize()).dt.days

In [13]:
user_features = merged_df.groupby('passport_id').agg(
    opening_adverts_amount=('advert_id', 'count'),
    price=('price', 'mean'),
    auto_age=('auto_age', 'mean'),
    advert_age=('advert_age', 'mean')
).reset_index()

median_auto_age = round(user_features['auto_age'].median())
max_advert_age = round(user_features['advert_age'].max())

print(f"Ответ: {median_auto_age}, {max_advert_age}")

Ответ: 14, 3385


### Задача 2. Признаки пользователя (3/5)
#### platform
Добавим в агрегацию столбец с основной платформой пользователя. Вдруг это тоже влияет на отток.

Если нельзя однозначно определить наиболее популярную платформу, то отдавайте предпочтение android > ios > desktop  > mobile > unknown (т.е. если одно объявление с android и одно с desktop, то основная платформа — android).

Создайте такой признак, назовите его `platform`.

Постарайтесь решить самостоятельно. Но если возникнут проблемы, один из вариантов решения — в подсказках ниже.

In [14]:
platform_priority = ['android', 'ios', 'desktop', 'mobile', 'unknown']

# Функция для определения основной платформы
def get_main_platform(platform_series):
    modes = platform_series.mode()
    
    if len(modes) == 1:
        return modes[0]
    
    for platform in platform_priority:
        if platform in modes.values:
            return platform
    
    return modes.iloc[0]

user_features = merged_df.groupby('passport_id').agg(
    opening_adverts_amount=('advert_id', 'count'),
    price=('price', 'mean'),
    auto_age=('auto_age', 'mean'),
    advert_age=('advert_age', 'mean'),
    platform=('platform', get_main_platform)
).reset_index()

print("Распределение платформ:")
print(user_features['platform'].value_counts())

Распределение платформ:
android    2250
ios        1042
desktop     148
unknown       1
Name: platform, dtype: int64


### Задача 2. Признаки пользователя (4/5)
#### is_top_model
Создадим признак "есть ли среди объявлений пользователя машины, из топ-10 моделей". Идея признака в том, что популярные, "ходовые" товары обычно продаются быстрее.

В датафрейме, который мы агрегируем, найдите 10 наиболее часто встречаемых `auto_model`, не считая `Unknown model`.

Для каждого объявления посчитайте, входит является ли оно продажей модели из топ-10 (1 если да, 0 если нет).

Подумайте, какая функция агрегации тут подойдёт, если нас интересует, есть ли у пользователя хотя бы одно такое объявление.

Полученный признак назовите `is_top_model`.

---

У скольки пользователей значение признака равно 1?

In [15]:
top_models = merged_df[merged_df['auto_model'] != 'Unknown model']['auto_model'].value_counts().head(10).index

print("Топ-10 моделей:")
print(top_models.tolist())
print()

merged_df['is_top_model_individual'] = merged_df['auto_model'].isin(top_models).astype(int)

user_features = merged_df.groupby('passport_id').agg(
    opening_adverts_amount=('advert_id', 'count'),
    price=('price', 'mean'),
    auto_age=('auto_age', 'mean'),
    advert_age=('advert_age', 'mean'),
    platform=('platform', get_main_platform),
    is_top_model=('is_top_model_individual', 'max')  # max = хотя бы одно объявление с 1
).reset_index()

users_with_top_model = user_features['is_top_model'].sum()

print(f"Количество пользователей с is_top_model = 1: {users_with_top_model}")

Топ-10 моделей:
['Camry', '2110 (седан)', 'Passat', '2114 (хэтчбек)', 'ГАЗель', 'Priora 2170 (седан)', 'Granta 2190 (седан)', '2121 Нива', 'Priora 2172 (хэтчбек)', '2115 (седан)']

Количество пользователей с is_top_model = 1: 945


### Задача 2. Признаки пользователя (5/5)
Осталось несколько финальных штрихов в подготовке датафрейма с пользователями.

#### user_type_cars_name
Перенесём сюда `user_type_cars_name` из объявлений. Тут никаких хитростей: т.к. мы проджойнивали этот столблец по `passport_id` из другой таблицы, для каждого `passport_id` будет только одно значение `user_type_cars_name`. Подумайте, какая функция агрегации здесь будет наиболее очевидной.

#### churn
Хоть это и не признак, но не забудем при группировке добавить таргет. Тут тоже всё просто.

#### Ещё скрытые пропуски
Как мы видели до этого, в столбце с годом были значения 0. Вроде бы не пропуск, `.isna()` не поймает. Но по факту пропущенное значение.

В категориальных столбцах такое тоже может быть:

1. В столбце `user_type_cars_name` есть значение '' (пустая строка). Давайте заменим его на `np.nan`, чтобы явно указать, что мы рассматриваем его как пропуск. Из-за некоторых особенностей `pandas` чуть проще это сделать уже после группировки и агрегации (иначе может появиться значение `None` и потребуется чуть больше шагов). Если вдруг вы сохраняли какой-то из датафреймов в `csv` и читали из него, у вас может уже быть `nan`, тогда ничего делать не нужно.
2. В столбце `platform` есть значение `unknown`. В другой ситуации мы могли бы оставить "unknown" отдельной категорией. Но поскольку строка с таким значением только одна, удобнее будет рассматривать его как пропуск. Замените это значение на np.nan в получившемся после группировке датафрейме.

Готово. У нас есть датафрейм с пользователями. В нём должно получиться 9 столбцов (включая `passport_id` и таргет) и столько строк, сколько было уникальных пользователей в датасете с объявлениями.

При группировке данные должны были автоматически отсортироваться. Но на всякий случай проверьте, что датафрейм отсортирован по возрастанию passport_id (например, с помощью `.is_monotonic_increasing`). Это для воспроизводимости.

Посмотрите, сколько получилось пользователей с разным типом `user_type_cars_name`.

Какой наименее частый тип `user_type_cars_name`? Сколько там пользователей?

---
Введите название (без кавычек) и количество пользователей через запятую и пробел. 

Например, `cars_seller`, `1200`.

In [16]:
user_features = merged_df.groupby('passport_id').agg(
    opening_adverts_amount=('advert_id', 'count'),
    price=('price', 'mean'),
    auto_age=('auto_age', 'mean'),
    advert_age=('advert_age', 'mean'),
    platform=('platform', get_main_platform),
    is_top_model=('is_top_model_individual', 'max'),
    user_type_cars_name=('user_type_cars_name', 'first'),
    churn=('churn', 'first')
).reset_index()

user_features.head()

,passport_id,opening_adverts_amount,price,auto_age,advert_age,platform,is_top_model,user_type_cars_name,churn
0,123467852,1,1900000.0,19.0,11.00,android,0,cars_simple,0
1,123469393,1,1000.0,<NA>,59.00,ios,0,cars_simple,0
2,123469843,4,1720000.0,16.0,660.25,desktop,0,cars_simple,0
3,123475067,2,10250000.0,7.5,632.00,android,0,cars_simple,0
4,123476026,1,2400000.0,14.0,2772.00,desktop,0,cars_simple,0


In [17]:
user_features['user_type_cars_name'] = user_features['user_type_cars_name'].replace('', np.nan)
user_features['platform'] = user_features['platform'].replace('unknown', np.nan)

print(f"Датафрейм отсортирован по возрастанию passport_id: {user_features['passport_id'].is_monotonic_increasing}")

user_type_counts = user_features['user_type_cars_name'].value_counts()
print("\nРаспределение user_type_cars_name:")
print(user_type_counts)

least_common_type = user_type_counts.index[-1]
least_common_count = user_type_counts.iloc[-1]

print()
print(f"Наименее частый тип: {least_common_type}, {least_common_count}")

Датафрейм отсортирован по возрастанию passport_id: True

Распределение user_type_cars_name:
cars_simple    2970
cars_seller     321
cars_dealer      37
Name: user_type_cars_name, dtype: int64

Наименее частый тип: cars_dealer, 37


### Задача 3. Подготовка к обучению (1/7)
В качестве метрики будем использовать `precision` при условии `recall > 0.8`. Бизнес хочет, чтобы когда мы предсказываем, какие клиенты утекут, мы "ловили" не менее 80% тех, кто действительно утечёт. При этом, конечно, хочется, чтобы было меньше ложных срабатываний, т.е. чтобы была как можно более высокая точность.

Как только установлена метрика, всегда хорошая идея прикинуть, какие у нас бейслайны. Какие наилучшие значения метрик можно получить, не обучая никаких моделей, а взяв простейшее решение вроде "всегда предсказываем 0", "всегда предсказываем 1", "всегда предсказываем самый популярный класс" и т.д.

Без этого сложно понимать, хорошо ли мы справились. Например, значение accuracy, равное 0.9 может быть как очень хорошим, так и очень плохим результатом, в зависимости от данных и задачи.

Прежде чем разбираться конкретно с нашим случаем, рассмотрим гипотетический пример.

Предположим, что в таргете 30% единичек, т.к. отток случается в 30% случаев. Какие максимальные значения метрик можно получить с помощью простейших наивных подходов, описанных выше?

В поле ниже введите через запятую и пробел:

- Какого значения `accuracy` можно добиться, не обучая никакую модель.
- Какого значения `recall` можно добиться, не обучая никакую модель.
- Какого значения `precision` можно добиться, не обучая никакую модель.
- Какого значения `precision` при условии `recall > 0.8` можно добиться, не обучая никакую модель.

---
Например, `0.3, 0.7, 1, 0`.

---
### Ответ:

`accuracy`. Тут можно возвращать самый популярный класс, в данном случае 0. Говорим, что никто не утечёт, в 70% случаев угадываем, что дейстивительно не утекут.

`recall`. Тут можно всегда возвращать единичку. Говорите, что утекут все. Предсказание захватывает вообще всех клиентов, в том числе всех, которые действительно утекут. Значение метрики равно `1`, вы великолепны.

Из-за этого `recall` обычно рассматривается не как основная и единственная метрика, а в связке с чем-то ещё.

`precision`. Тут тоже можно всегда возвращать `1`. Говорим, что утекут все. В 30% случаев угадываем. Чего-то лучше без модели не добиться.

`precision` при условии `recall > 0.8`. Тут ничего не меняется. Говорим, что утекут все. Получаем `recall` равный 1 (условие выполняется: `1 > 0.8`) и `precision = 0.3`. Как-то использовать имеющийся "запас" `recall` не получится. Если, например, предсказывать, что утекут случайные 80%, получим `recall = 0.8` и тот же `precision = 0.3` (не считая небольшого случайного колебания).

Помимо понимания сути метрик, важный результат здесь в том, что доля класса 1 в данных будет бейслайном по качеству в нашей задаче.

### Задача 3. Подготовка к обучению (2/7)
#### X и y
Разбейте датафрейм на X и y. В X пойдут все столбцы кроме `passport_id` и таргета.

#### Train и test
Разбейте данные на `train` и `test`. Используйте `test_size=0.2`, `random_state=42`. Также, поскольку это задача классификации, добавьте параметр `stratify=y`, чтобы в `train` и `test` получились такие же пропорции классов, какие были до разбиения.

Посмотрите, какие доли классов были в исходном датафрейме. Убедитесь, что в `train` и `test` получились примерно такие же.

Как обсуждали на предыдущем шаге, это значение также задаёт бейслайн значения оптимизируемой метрики.

Какова доля класса 1 в каждом из датасетов? Округлите значение до 3 знаков после точки, например, 0.456. Введите одно число, т.к. с точностью до 3 знаков значение будет одинаковым для каждой из выборок.

In [18]:
X = user_features.drop(['passport_id', 'churn'], axis=1)
y = user_features['churn']

original_churn_rate = y.mean()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

train_churn_rate = y_train.mean()
test_churn_rate = y_test.mean()

print(f"Доля класса 1 в train: {train_churn_rate:.3f}")
print(f"Доля класса 1 в test: {test_churn_rate:.3f}")

Доля класса 1 в train: 0.504
Доля класса 1 в test: 0.504


### Задача 3. Подготовка к обучению (3/7)
#### Пайплайн предподготовки
Напишем пайплайн предподготовки данных.

В целом, тут возможны два подхода:

1. Пишем отдельно пайплайн со всеми шагами для числовых столбцов и отдельно пайплайн со всеми шагами для категориальных. Потом объединяем их через ColumnTransformer в единый трансформер предподготовки.
2. Пишем единый пайплайн предподготовки, с общими шагами вроде "заполнение пропусков", "кодирование категориальных данных", "масштабирование признаков". При необходимости ветвимся внутри каждой стадии с помощью ColumnTransformer, чтобы по-разному обработать значения разных типов.

Оба подхода хорошие, тут скорее вопрос стиля (и того, что больше подойдёт в конкретной задаче). Мы будем использовать второй подход. Но имейте в виду и первый, где-то он может оказаться более простым и наглядным.

Напишите пайплайн предподготовки данных, состоящий из 3 этапов:

(1) Заполняем пропуски. Тут будет три случая:

- Пропуски в столбце `platform` заполняем самым частым значением. Пропусков было мало, важной информации они явно не несут. Какая-то основная платформа у пользователя точно есть, мы  просто не знаем, какая. Просто вливаем в самую большую категорию, считая, что она наименее информативна.
- Пропуски в столбце `user_type_cars_name` заполним фиксированным значением 'unknown'. Пропусков в этом столбце много, столбец может оказаться важным. Лучше сохранить информацию, что значения были пустыми, пусть это будет отдельной категорией.
- Пропуски в числовых столбцах заменяем средним значением.

Добавьте параметры `remainder='passthrough'` и `verbose_feature_names_out=False` в `ColumnTransformer`, чтобы мы не потеряли остальные столбцы, а имена столбцов были читаемыми.

(2) Кодируем категориальные данные.

В наших данных у всех категорий небольшое количество уникальных значений. Так что просто берём OneHotEncoder, без каких-то хитростей и условий. Добавьте параметр `drop='first'`, т.к. мы планируем использовать линейную модель, а также `handle_unknown='ignore'`, `sparse_output=False`.

Не забудьте те же параметры `ColumnTransformer`, которые обсуждали в предыдущем пункте.

(3) Масштабируем признаки.

Применим `StandardScaler` ко всем столбцам.

Для всего пайплайна целиком поставьте `.set_output(transform='pandas')`, чтобы нам было удобнее работать с результатом преобразования.

Использовать этот пайплайн мы будем как часть пайплайна обучения модели. Но для проверки давайте вызовем `fit_transform` и посмотрим, что получается.

Примените пайплайн к обучающей выборке. Сохраните результат в csv с параметром `index=False` и отправьте на проверку.

In [19]:
prep = Pipeline([
    ('imputer', ColumnTransformer([
        ('cat_mode', SimpleImputer(strategy='most_frequent'), ['platform']),
        ('cat_unk', SimpleImputer(strategy='constant', fill_value='unknown'), ['user_type_cars_name']),
        ('num', SimpleImputer(strategy="mean"), selector(dtype_include='number')),
    ], remainder='passthrough', verbose_feature_names_out=False)),
    
    ('encoder', ColumnTransformer([
        ('cat', OneHotEncoder(
            drop='first', 
            handle_unknown='ignore', 
            sparse_output=False
        ), selector(dtype_exclude='number'))
    ], remainder='passthrough', verbose_feature_names_out=False)),
    
    ('scaler', StandardScaler())
]).set_output(transform='pandas')

In [20]:
prep_result = prep.fit_transform(X_train, y_train)
prep_result

,platform_desktop,platform_ios,user_type_cars_name_cars_seller,user_type_cars_name_cars_simple,user_type_cars_name_unknown,opening_adverts_amount,price,auto_age,advert_age,is_top_model
2682,-0.219957,1.532296,-0.323693,0.404520,-0.193174,-0.202769,-0.596921,0.068329,1.491521,-0.616555
6,4.546349,-0.652615,3.089346,-2.472066,-0.193174,-0.202769,1.160889,-1.508082,-0.203416,1.621915
2649,-0.219957,1.532296,-0.323693,0.404520,-0.193174,-0.202769,-0.613715,0.189591,-0.319580,-0.616555
2590,-0.219957,-0.652615,3.089346,-2.472066,-0.193174,-0.202769,-0.511550,-0.052933,-0.118934,-0.616555
3032,-0.219957,-0.652615,-0.323693,0.404520,-0.193174,-0.202769,-0.168665,0.432116,-0.330141,-0.616555
...,...,...,...,...,...,...,...,...,...,...
518,-0.219957,-0.652615,-0.323693,0.404520,-0.193174,-0.202769,-0.560533,0.310854,-0.250938,1.621915
536,-0.219957,1.532296,-0.323693,0.404520,-0.193174,-0.202769,-0.294622,-0.659245,-0.303740,1.621915
1400,-0.219957,1.532296,-0.323693,0.404520,-0.193174,-0.202769,-0.697687,0.000000,1.364796,-0.616555
2517,-0.219957,-0.652615,-0.323693,0.404520,-0.193174,-0.202769,-0.392589,1.644740,-0.155895,-0.616555


In [21]:
prep_result.to_csv('prep_result.csv', index=False)

### Задача 3. Подготовка к обучению (4/7)
#### Пайплайн
Напишем пайплайн модели, состоящий из уже готового пайплайна предподготовки и LogisticRegression.

Для `LogisticRegression` поставьте параметры `random_state=42` и `solver='liblinear'`. Дело в том, что дефолтный солвер `lbfgs` поддерживает только `l2`-регуляризацию, а мы хотим попробовать и `l2`, и `l1`.

---
#### Вспомним идею использования threshold
Как обсуждалось в лекции, логистическая регрессия в качестве результата может выдавать вероятности и значения классов. Значения классов получаются на основе предсказанных вероятностей: если вероятность отнесения к классу 1 больше 0.5, то относим к классу 1, иначе к классу 0.

Несмотря на то, что это звучит разумно, часто может быть полезным взять другой порог: не 0.5, а, например, 0.4 или 0.6. И это позволит улучшить значения метрик в нужном направлении.

Пусть, например, модель предсказала вероятности `0.2, 0.4, 0.6, 0.8`. По умолчанию это превратится в классы `0, 0, 1, 1`.

Мы можем взять порог 0.3: всё, что получило вероятность более 0.3, будет внесено в класс 1. Получатся классы `0, 1, 1, 1`. По второму элементу мы были уверены только на 40%, но всё равно "рискнули" и сказали, что он утечёт. Скорее всего, это повысит recall (поймаем больше тех, кто правда утечёт), но понизит precision (будет много ложных срабатываний).

И наоборот: можем взять порог 0.7, тогда получим классы `0, 0, 0, 1`. Мы решили, что даже уверенности на 60% нам недостаточно, чтобы утверждать, что клиент утечёт, мы хотим быть уверены на 70%. Скорее всего, это повысит precision (т.к. относим к оттоку только тех, в ком точно уверены), но уронит recall (многих пропустим из-за этого).

---
Если модель не выдаёт вероятности, мы всё равно можем в каком-то смысле имитировать, что она их выдаёт, с помощью `CalibratedClassifierCV`. И дальше вся логика та же самая.

Threshold обычно не рассматривается как гиперпараметр модели при обучении, потому что он никак не влияет на процесс обучения самой модели. При любом пороге сама модель одинаковая, она будет выдавать одинаковые вероятности. Вопрос только в том, как мы эти вероятности будем использовать. Т.е. часто это настраивается уже после обучения модели, а не во время.

Но подход, используемый в лекции, тоже имеет свои плюсы: если мы сделаем threshold гиперпараметром модели, будет удобно пробовать разные пороги в grid search. Так что в этом задании попробуем как раз его.

### Задача 3. Подготовка к обучению (5/7)
#### ThresholdClassifier
В лекции были `LogRegClassifier` и `SVMClassifier`, которые являются обёртками для соответствующих моделей и позволяют задать threshold как гиперпараметр. Можно было бы скопипастить их оттуда и использовать здесь. Но! Давайте напишем реализацию сами, чтобы лучше разобраться, что и зачем мы делаем.

Чтобы не полностью повторяться, вместо двух классов напишем один универсальный. И поменяем некоторые детали.

Попробуйте выполнить это задание, не подглядывая в лекцию. Мы подробно распишем все шаги, которые нужно сделать. Это задание без проверки, решение будет на следующем степе, так что не беспокойтесь, если что-то не получится.

---
Итак, реализуйте класс `ThresholdClassifier`, который позволяет добавить в модель threshold в качестве гиперпараметра.

Класс будет наследоваться от `BaseEstimator` и `ClassifierMixin` (оба из sklearn.base).

---
В `__init__` принимаем два параметра: `base_model` с базовой моделью (или пайплайном) и threshold со значением по умолчанию 0.5. Сохраняем аргументы в поля класса.

---
Метод `fit(self, X, y)` делает три вещи:

- Проверяем, есть ли у базовой модели метод `predict_proba`. Проверить можно с помощью `hasattr`. Если такого атрибута нет, значит модель не умеет сама предсказывать вероятности. Тогда оборачиваем эту базовую модель в `CalibratedClassifierCV` (из `sklearn.calibration`) и сохраняем результат обратно в `self.base_model`. Т.е. заменяем "просто модель" на модель, обёрнутую в `CalibratedClassifierCV`. Теперь модель умеет предсказывать вероятности. Для `CalibratedClassifierCV` поставьте параметр `cv=3` (чтобы побыстрее училось). А параметр method задавать не будем, отставим значение по умолчанию.
- Вызываем `fit` для `base_model`.
- После обучения берём из `base_model` классы `classes_` и сохраняем их в поле с таким же именем (это нужно для работы grid search).

---
Метод `predict_proba(self, X)` просто вызывает аналогичный метод базовой модели и возвращает результат.

---
Метод `predict(self, X)` — то, ради чего всё затевалось. Но в нём будет буквально пара строк:

- Вызываем `predict_proba`. Из результата берём вероятности для класса 1: [:, 1].
- Считаем для этих вероятностей `>= self.threshold` и приводим результат к целому. Это как раз будет класс 1 там, где порог преодолён, и класс 0 там, где нет. Полученный массив возвращаем из метода.

---
Потом можно создать экземпляр и, например, посмотреть, как меняется процент единичек в зависимости от заданного threshold:

```python
logreg_with_threshold = ThresholdClassifier(logreg_pipe, threshold=0.5)
logreg_with_threshold.fit(X_train, y_train).predict(X_train).mean()
```

Тут мы предсказываем на том же `X_train`, на котором учимся, но поскольку это просто для проверки работы класса, это ок.

In [22]:
class ThresholdClassifier(BaseEstimator, ClassifierMixin):
    """Wrapper for classifiers that allows threshold adjustment"""
    def __init__(self, base_model, threshold=0.5):
        self.base_model = base_model
        self.threshold = threshold
    
    def fit(self, X, y):
        if not hasattr(self.base_model, 'predict_proba'):
            self.base_model = CalibratedClassifierCV(self.base_model, cv=3)
        self.base_model.fit(X, y)
        self.classes_ = self.base_model.classes_
        return self
    
    def predict_proba(self, X):
        return self.base_model.predict_proba(X)
    
    def predict(self, X):
        y_pred_proba = self.predict_proba(X)[:, 1]
        return (y_pred_proba >= self.threshold).astype(int)

### Задача 3. Подготовка к обучению (6/7)
#### Refit
Мы хотим использовать нестандартную метрику: оптимизируем `precision` при условии `recall > 0.8`. Что передавать в grid search в качестве scoring?

Один из возможных вариантов — написать кастомную метрику, которая будет возвращать `precision` только при выполнении условия по `recall`. А если условие не выполняется, возвращать 0 или минус бесконечность.

У такого подхода есть минус. Если мы будем использовать такую метрику в grid search и на одном из фолдов получится `recall` чуть ниже 0.8, это испортит нам метрику. В идеале мы бы хотели чтобы учитывался средний `recall` по всем фолдам, т.к. это даст более устойчивый результат.

Решение — задать кастомный `refit`. В качестве метрик используем обычные `precision` и `recall`, а вот на этапе определения того, какая модель справилась лучше всего, — добавляем кастомную логику.

В лекции был пример реализации `refit_strategy`. Но давайте напишем свою функцию. Во-первых, опять же, чтобы попрактиковаться и лучше разобраться. Во-вторых, мы посмотрим, как это можно сделать иначе. Буквально в несколько строк кода.

---
Напишем функцию `refit_with_threshold`. Шаблон реализации:
```python
def make_refit_with_recall_threshold(recall_threshold):
    """
    Returns a refit function that selects the model with the highest precision
    if recall exceeds the given threshold.
    """
    def refit_with_threshold(cv_results):
        # Ваш код здесь
    
    return refit_with_threshold
```

Можете пока не обращать внимания на обёртку `make_refit_with_recall_threshold` (она нужна чтобы было удобнее задать порог recall'а) и сосредоточиться на самой `refit_with_threshold`.

Что функция должна делать:

1. Преобразуем `cv_results` в pandas-датафрейм. Теперь результаты grid search у нас в удобном виде. Каждая строка это определённая комбинация гиперпараметров, которая пробовалась. Столбцы — значения гиперпараметров, метрик и т.д. Нам нужно будет выдать индекс строки датафрейма с максимальным precision при выполнении условия на `recall`.
2. Оставляем только строки, в которых значение столбца `mean_test_recall` больше, чем `recall_threshold`.
3. Если датафрейм получился пустым, значит ни одной такой комбинации нет. Тут нам остаётся только сделать `raise ValueError` с текстом ошибки.
4. Если всё ок, то возвращаем индекс строки с максимальным значением столбца `mean_test_precision` (можно использовать `.idxmax()`).

---
Пример использования:

```python
refit = make_refit_with_recall_threshold(recall_threshold=0.8)
```

Этот код выдаст функцию `refit_with_threshold`, в которую будет вшит параметр `recall_threshold=0.8`. Дальше её можно будет использовать в grid search.

Это задание без проверки, на следующем степе будет ответ. Но постарайтесь сделать самостоятельно, т.к. это хорошая практика, которая поможет вам лучше разобраться в том, как использовать grid search.

In [23]:
def make_refit_with_recall_threshold(recall_threshold):
    """
    Returns a refit function that selects the model with the highest precision
    if recall exceeds the given threshold.
    """
    def refit_with_threshold(cv_results):
        results_df = pd.DataFrame(cv_results)
        valid_models = results_df[results_df['mean_test_recall'] > recall_threshold]
        if valid_models.empty:
            raise ValueError(f'No models met the recall threshold of {recall_threshold}.')
        return valid_models['mean_test_precision'].idxmax()
    
    return refit_with_threshold

#### Метрики
Поскольку у нас теперь есть кастомный `refit`, в качестве метрик можно взять обычные `precision` и `recall`. То есть передавать в grid search `scoring=['precision', 'recall']`. А о том, чтобы выполнялось условие на `recall`, позаботится `refit`: в `best_estimator_` гарантировано будет модель, которая прошла порог.

Небольшая проблема в том, что мы теперь можем как угодно крутить threshold. И если мы его выкрутим очень неудачно, так, что в предсказании получатся все нули, `precision` может ругаться, что не может на этом посчитаться.

Чтобы этого избежать, давайте явно зададим, что в таком случае можно просто возвращать 0:

In [24]:
scoring = {
    'precision': make_scorer(precision_score, zero_division=0),
    'recall': 'recall'
}

#### Сплиттер
Последнее, что нужно сделать перед обучением, — подготовить сплиттер для разбиения на фолды.

Создайте `StratifiedKFold` с параметрами `n_splits=3`, `shuffle=True`, `random_state=42`.

Как и при разбиении на `train` и `test`, используем стратифицированную версию (`StratifiedKFold`, а не просто `KFold`), чтобы сохранить пропорции классов.

Фух, теперь всё. Переходим к обучению моделей.

In [25]:
splitter = StratifiedKFold(
    n_splits=3,      
    shuffle=True,
    random_state=42
)

### Задача 4. Обучение и выбор модели (1/4)
#### Логистическая регрессия: пробный запуск
У нас есть пайплайн для логистической регрессии, сплиттер, scoring, функция refit. Cоберём всё вместе и запустим grid search.

Для начала давайте запустим на грубой, "разведочной", сетке гиперпараметров, где широкими мазками обозначены два из них:

- Значение гиперпараметра регуляризации C: `0.0001, 0.01, 1, 100, 10000`.
- threshold: `0.4, 0.5, 0.6`.

Смотреть будем не на то, какие гиперпараметры получились самыми лучшими и чему равна метрика, а на закономерности в целом. Чтобы понять, какие значения есть смысл перебирать подробнее.

Запустите grid search на обучающей выборке и посмотрите на таблицу с результатами в `.cv_results_`. Удобнее всего преобразовать её в датафрейм и отфильтровать лишнее. Например, так (замените на  свои имена):

```python
cols = [
    'param_base_model__logreg__C',
    'param_threshold',
    'mean_test_precision',
    'mean_test_recall',
]

pd.DataFrame(search_logreg.cv_results_)[cols]
```

Что можно сказать о результатах?

In [26]:
logreg_pipe = Pipeline(
    [
        ('preprocessing', prep),
        ('logreg', LogisticRegression(solver='liblinear', random_state=42))
    ]
)

In [27]:
logreg_with_threshold = ThresholdClassifier(logreg_pipe, threshold=0.5)
logreg_with_threshold.fit(X_train, y_train).predict(X_train).mean()

0.6726017441860465

In [28]:
refit = make_refit_with_recall_threshold(recall_threshold=0.8)

In [29]:
logreg_grid_rough = {
    'base_model__logreg__C' :  [0.0001, 0.01, 1, 100, 10000],
    'threshold': [0.4, 0.5, 0.6],
}

In [30]:
%%time

search_logreg = GridSearchCV(
    logreg_with_threshold, logreg_grid_rough, 
    cv=splitter, scoring=scoring, refit=refit
)

search_logreg.fit(X_train, y_train)

CPU times: user 3.52 s, sys: 7.34 ms, total: 3.53 s
Wall time: 3.6 s


GridSearchCV(cv=StratifiedKFold(n_splits=3, random_state=42, shuffle=True),
             estimator=ThresholdClassifier(base_model=Pipeline(steps=[('preprocessing',
                                                                       Pipeline(steps=[('imputer',
                                                                                        ColumnTransformer(remainder='passthrough',
                                                                                                          transformers=[('cat_mode',
                                                                                                                         SimpleImputer(strategy='most_frequent'),
                                                                                                                         ['platform']),
                                                                                                                        ('cat_unk',
                                                                                                                         SimpleImputer(fill_value='unknown',
                                                                                                                                       strategy...
                                                                                        StandardScaler())])),
                                                                      ('logreg',
                                                                       LogisticRegression(random_state=42,
                                                                                          solver='liblinear'))])),
             param_grid={'base_model__logreg__C': [0.0001, 0.01, 1, 100, 10000],
                         'threshold': [0.4, 0.5, 0.6]},
             refit=<function make_refit_with_recall_threshold.<locals>.refit_with_threshold at 0x7f56334c3d30>,
             scoring={'precision': make_scorer(precision_score, zero_division=0),
                      'recall': 'recall'})

In [31]:
cols = [
    'param_base_model__logreg__C',
    'param_threshold',
    'mean_test_precision',
    'mean_test_recall',
]

pd.DataFrame(search_logreg.cv_results_)[cols]

,param_base_model__logreg__C,param_threshold,mean_test_precision,mean_test_recall
0,0.0001,0.4,0.504360,1.000000
1,0.0001,0.5,0.572143,0.809773
2,0.0001,0.6,0.000000,0.000000
3,0.01,0.4,0.552220,0.923604
4,0.01,0.5,0.585529,0.800405
5,0.01,0.6,0.695432,0.274490
6,1,0.4,0.569843,0.891903
7,1,0.5,0.593722,0.795362
8,1,0.6,0.685969,0.337940
9,100,0.4,0.570559,0.892623


### Задача 4. Обучение и выбор модели (2/4)
#### Логистическая регрессия: подбираем гиперпараметры
Исходя из результатов предыдущего шага, попробуем перебрать такие значения гиперпараметров:

- C: `0.1, 1, 10, 100, 1000`.
- threshold: `0.42, 0.43, 0.44, 0.45, 0.46, 0.47, 0.48, 0.49, 0.50, 0.51`. Скорее всего, нужно что-то чуть меньше 0.5, но возьмём с запасом в обе стороны, больше в сторону 0.4.
- penalty: `'l1'`, `'l2'`.

Запустите grid search на этой сетке.

Как вы можете увидеть, у результата есть `.best_params_` и `.best_estimator_`, но нет `.best_score_` (хотя обычно такое свойство имеется). Из-за того, что у нас несколько метрик в scoring и ещё и кастомный refit, grid search не знает, какую метрику мы рассматриваем в качестве основной (и нет прямого способа её задать).

Но это просто обойти. Будем брать её из таблички с результатами по `.best_index_`. Вызовите такую функцию сразу после `fit`:

```python
def set_best_score(grid_search):
    grid_search.best_score_ = grid_search.cv_results_['mean_test_precision'][grid_search.best_index_]
```

Например,

```python
search_logreg.fit(X_train, y_train)
set_best_score(search_logreg)
```

Тогда `.best_score_` появится.

Ещё вам может быть полезна такая функция, которая выдаёт результаты в виде отфильтрованного и отсортированного датафрейма:

```python
def cv_results_dataframe(grid, recall_threshold=0.8, max_rows=10):
    cols = (
        [k for k in grid.cv_results_.keys() if k.startswith('param_')] +
        ['mean_test_precision', 'mean_test_recall']
    )
    res = pd.DataFrame(grid.cv_results_)[cols]
    res = res[res['mean_test_recall'] > recall_threshold]
    return res.sort_values('mean_test_precision', ascending=False).head(max_rows)
```

Оставляем только столбцы с параметрами и две метрики, преобразуем в датафрейм, фильтруем по `recall`, сортируем по `precision`, оставляем 10 лучших строк.

Посмотрите не только то, какая комбинация гиперпараметров сработала лучше всего, но и какие другие тоже показали себя неплохо.

Но вопрос этого степа будет всё же про то, что сработало лучше всего. Получилось ли у нас существенно обойти бейслайн?

Через запятую и пробел введите 4 значения:

- Полученное значение метрики `precision` (округлите до 3 знаков).
- Подобранное значение гиперпараметра `C`.
- Значение `penalty` (без кавычек).
- `threshold`.

Например, `0.543, 1000, l1, 0.48`.

`Recall` вводить не нужно (т.к. нас не особо интересует конкретное значение). Но проверьте по табличке, что он действительно выше 0.8.

In [32]:
def set_best_score(grid_search):
    grid_search.best_score_ = grid_search.cv_results_['mean_test_precision'][grid_search.best_index_]

logreg_grid_rough = {
    'base_model__logreg__C': [0.01, 1, 10, 100, 1000],
    'base_model__logreg__penalty': ['l1', 'l2'],
    'threshold': [0.42, 0.43, 0.44, 0.45, 0.46, 0.47, 0.48, 0.49, 0.50, 0.51]
}

In [33]:
%%time

search_logreg = GridSearchCV(
    logreg_with_threshold, logreg_grid_rough, 
    cv=splitter, scoring=scoring, refit=refit
)

search_logreg.fit(X_train, y_train)
set_best_score(search_logreg)

CPU times: user 22 s, sys: 20.7 ms, total: 22 s
Wall time: 22 s


In [34]:
def cv_results_dataframe(grid, recall_threshold=0.8, max_rows=10):
    cols = (
        [k for k in grid.cv_results_.keys() if k.startswith('param_')] +
        ['mean_test_precision', 'mean_test_recall']
    )
    res = pd.DataFrame(grid.cv_results_)[cols]
    res = res[res['mean_test_recall'] > recall_threshold]
    return res.sort_values('mean_test_precision', ascending=False).head(max_rows)

In [35]:
cv_results_dataframe(search_logreg)

,param_base_model__logreg__C,param_base_model__logreg__penalty,param_threshold,mean_test_precision,mean_test_recall
57,10,l2,0.49,0.592511,0.811211
47,10,l1,0.49,0.592487,0.811210
97,1000,l2,0.49,0.592283,0.810490
87,1000,l1,0.49,0.592283,0.810490
77,100,l2,0.49,0.592283,0.810490
67,100,l1,0.49,0.592283,0.810490
37,1,l2,0.49,0.591376,0.810490
27,1,l1,0.49,0.590870,0.811931
46,10,l1,0.48,0.586488,0.824178
96,1000,l2,0.48,0.586287,0.823458


In [36]:
precision = round(cv_results_dataframe(search_logreg).mean_test_precision.head(1).values[0], 3)
C = cv_results_dataframe(search_logreg).param_base_model__logreg__C.head(1).values[0]
penalty = cv_results_dataframe(search_logreg).param_base_model__logreg__penalty.head(1).values[0]
threshold = cv_results_dataframe(search_logreg).param_threshold.head(1).values[0]

print(f"Ответ: {precision}, {C}, {penalty}, {threshold}")

Ответ: 0.593, 10, l2, 0.49


### Задача 4. Обучение и выбор модели (4/4)
#### SVM (LinearSVC)
Попробуем вместо логистической регрессии использовать линейный SVM-классификатор.

Создайте пайплайн, состоящий из пайплайна предобработки и `LinearSVC` c параметрами `random_state=42`, `dual='auto'`.

Оберните пайплайн в `ThresholdClassifier`, чтобы превратить его в модель, в которой можно настраивать threshold как гиперпараметр.

Тут можно было бы тоже сначала прикинуть, где лучше искать гиперпарамеры. Но давайте сразу возьмём ту же сетку по threshold и по регуляризации. Если увидим, что какие-то гиперпараметры слишком "на краю", то поменяем сетку.

- C: `0.1, 1, 10, 100, 1000`.
- threshold: `0.42, 0.43, 0.44, 0.45, 0.46, 0.47, 0.48, 0.49, 0.50, 0.51`.

Запустите, получите лучшую метрику и лучшие гиперпараметры. Посмотрите табличку с несколькими лучшими результатами (что можно по ней сказать?).

В поле ниже через запятую и пробел введите 3 значения: `precision` (округлите до 3 знаков), `C`, `threshold`.

Например, `0.543, 1000, 0.48`.

Recall вводить не нужно (т.к. нас не особо интересует конкретное значение). Но проверьте по табличке, что он действительно выше 0.8.

In [37]:
svm_pipe = Pipeline(
    [
        ('preprocessing', prep),
        ('svm', LinearSVC(random_state=42, dual='auto'))
    ]
)

svm_with_threshold = ThresholdClassifier(svm_pipe)

In [38]:
svm_grid = {
    'base_model__svm__C' : [0.1, 1, 10, 100, 1000],
    'threshold': [0.42, 0.43, 0.44, 0.45, 0.46, 0.47, 0.48, 0.49, 0.50, 0.51],
}

In [39]:
%%time

search_svc = GridSearchCV(
    svm_with_threshold, svm_grid,
    cv=splitter, scoring=scoring, refit=refit
)

search_svc.fit(X_train, y_train)
set_best_score(search_svc)

CPU times: user 42 s, sys: 63.9 ms, total: 42.1 s
Wall time: 42.1 s


In [40]:
cv_results_dataframe(search_svc)

,param_base_model__svm__C,param_threshold,mean_test_precision,mean_test_recall
48,1000,0.5,0.592091,0.801838
28,10,0.5,0.592091,0.801838
38,100,0.5,0.592091,0.801838
18,1,0.5,0.591981,0.800398
8,0.1,0.5,0.590982,0.800397
27,10,0.49,0.589405,0.815530
37,100,0.49,0.589405,0.815530
47,1000,0.49,0.589405,0.815530
7,0.1,0.49,0.589347,0.813368
17,1,0.49,0.588972,0.814088


In [41]:
precision = round(cv_results_dataframe(search_svc).mean_test_precision.head(1).values[0], 3)
C = cv_results_dataframe(search_svc).param_base_model__svm__C.head(2).values[1]
threshold = cv_results_dataframe(search_svc).param_threshold.head(1).values[0]

print(f"Ответ: {precision}, {C}, {threshold}")

Ответ: 0.592, 10, 0.5


### Задача 5. Используем дополнительные данные (1/3)
#### Попробуем добавить информацию о транзакционной активности клиентов.

Кажется, что различного рода транзакции (продажи / покупки и пр.) могут быть хорошим сигналом для оттока. Если клиент часто и объемно транзачит, то, скорее всего, с нашим сервисом его связывает больше вещей, чем продавцов, которые не пользуются премиальными услугами, не возобновляют подписки и так далее.

Посмотрите на клиентские транзакции в таблице `user_transactions` за период между `2022-09-01` и `2022-10-01` включительно (вспомним таймлайн: мы хотим из 1 октября предсказать, что случится 1 ноября; так что берём транзакции за предыдущий месяц, сентябрь).

Объединим типы транзакций в более общие классы:

1) Транзакции класса revenue, если `type in ('basic sale', 'fast sale', 'quick sale')`

2) Транзакции класса renewal, если `type in ('renewal', 'archived renewal', 'automatic renewal')`

3) Транзакции класса premium, если `type in ('premium', 'premium weekly')`

Для каждого `passport_id` просуммируйте amount по каждому из 3 классов транзакций. Перед суммами добавьте минус, чтобы результат получился с правильным знаком. Полученные столбцы назовите по классам транзакций: revenue, renewal, premium.

Т.к. мы работаем с Clickhouse, можно использовать удобные функции вроде sumIf (но конечно можно использовать и классический SQL).

Приджойните результаты к нашему датафрейму (который был до разбиения на X/y и train/test). Подумайте, какой тип джойна тут нужен и почему.

Заполните пропуски нулями (только в этих 3 столбцах). Мы это делаем потому, что по смыслу это не пропуски (которые "на самом деле" ненулевые, просто у нас нет о них информации), а именно что отсутствие транзакций, нулевые суммы.

Если посмотреть на данные в 3 новых столбцах, видно, что очень просится логарифм, чтобы сделать их более нормальными (тем более учитывая, что мы используем линейные модели). Небольшая проблема в том, что в данных есть нули, а логарифм нуля не определён. Один из вариантов — сначала добавить единичку, а потом взять логарифм. Тогда нули останутся на месте. Прибавьте 1 к значениям трёх новых столбцов и потом прологарифмируйте.

Даже после преобразования данные всё ещё сильно скошенные, но, возможно, логарифмирование всё же поможет моделям.

Посчитайте средние значения для каждого из 3 новых столбцов, округлив до 2 знаков после точки.

Введите значения через запятую и пробел в таком порядке: `revenue`, `renewal`, `premium`.

Например, `0.12, 3.45, 6.78.`

In [42]:
transaction_data_query = '''
    SELECT passport_id,
        -sumIf(amount, type in ('basic sale', 'fast sale', 'quick sale')) as revenue,
        -sumIf(amount, type in ('renewal', 'archived renewal', 'automatic renewal')) as renewal,
        -sumIf(amount, type in ('premium', 'premium weekly')) as premium
    FROM user_transactions
    WHERE payment_date BETWEEN '2022-09-01' AND '2022-10-01'
    GROUP BY passport_id
    ORDER BY passport_id
'''

transaction_data = get_data(transaction_data_query)

In [43]:
df4 = user_features.merge(transaction_data, how='left', on='passport_id')
df4[['revenue', 'renewal', 'premium']] = df4[['revenue', 'renewal', 'premium']].fillna(0)
df4[['revenue', 'renewal', 'premium']] = np.log(df4[['revenue', 'renewal', 'premium']] + 1)

df4

,passport_id,opening_adverts_amount,price,auto_age,advert_age,platform,is_top_model,user_type_cars_name,churn,revenue,renewal,premium
0,123467852,1,1900000.0,19.0,11.00,android,0,cars_simple,0,0.000000,0.000000,0.0
1,123469393,1,1000.0,<NA>,59.00,ios,0,cars_simple,0,0.000000,0.000000,0.0
2,123469843,4,1720000.0,16.0,660.25,desktop,0,cars_simple,0,0.000000,0.000000,0.0
3,123475067,2,10250000.0,7.5,632.00,android,0,cars_simple,0,0.000000,0.000000,0.0
4,123476026,1,2400000.0,14.0,2772.00,desktop,0,cars_simple,0,0.000000,6.634633,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
3436,144333736,1,520000.0,10.0,0.00,ios,0,cars_simple,0,7.738488,0.000000,0.0
3437,144333976,1,40000.0,19.0,0.00,ios,0,cars_simple,1,0.000000,0.000000,0.0
3438,144334504,1,84000.0,16.0,0.00,ios,0,cars_simple,1,0.000000,0.000000,0.0
3439,144334622,1,500000.0,23.0,0.00,android,0,cars_simple,1,0.000000,0.000000,0.0


In [44]:
revenue = df4[['revenue', 'renewal', 'premium']].mean().round(2)[0]
renewal = df4[['revenue', 'renewal', 'premium']].mean().round(2)[1]
premium = df4[['revenue', 'renewal', 'premium']].mean().round(2)[2]

print(f"Ответ: {revenue}, {renewal}, {premium}")

Ответ: 1.68, 2.44, 0.31


### Задача 5. Используем дополнительные данные (2/3)
Снова выделите из данных X, теперь с дополнительными признаками.

Разбейте его на `train` и `test` составляющие с теми же параметрами `stratify=y`, `test_size=0.2`, `random_state=42`.

y можно не трогать: он должен был остаться тем же (но перепроверьте на всякий случай, что это так).

Запустите grid search для логистической регрессии, так же, как до этого, и с той же сеткой гиперпараметров.

Получилось или улучшить качество?

Посмотрите табличку результатами по нескольким лучшим комбинациям гиперпараметров.

Через запятую и пробел введите 4 значения:

- Полученное значение метрики `precision` (округлите до 3 знаков).
- Подобранное значение гиперпараметра `C`.
- Значение `penalty` (без кавычек).
- `threshold`.

Например, `0.543, 1000, l1, 0.48`.

In [45]:
X = df4.drop(['passport_id', 'churn'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

In [46]:
%%time

search_logreg = GridSearchCV(
    logreg_with_threshold, logreg_grid_rough, 
    cv=splitter, scoring=scoring, refit=refit
)

search_logreg.fit(X_train, y_train)
set_best_score(search_logreg)

CPU times: user 58.8 s, sys: 1min 1s, total: 1min 59s
Wall time: 1min


In [47]:
cv_results_dataframe(search_logreg)

,param_base_model__logreg__C,param_base_model__logreg__penalty,param_threshold,mean_test_precision,mean_test_recall
34,1,l2,0.46,0.600408,0.803311
24,1,l1,0.46,0.600403,0.803309
94,1000,l2,0.46,0.599829,0.802588
84,1000,l1,0.46,0.599829,0.802588
74,100,l2,0.46,0.599829,0.802588
54,10,l2,0.46,0.599826,0.802588
64,100,l1,0.46,0.599615,0.801868
44,10,l1,0.46,0.599615,0.801868
33,1,l2,0.45,0.595857,0.820597
53,10,l2,0.45,0.595743,0.821317


In [48]:
precision = round(cv_results_dataframe(search_logreg).mean_test_precision.head(1).values[0], 3)
C = cv_results_dataframe(search_logreg).param_base_model__logreg__C.head(1).values[0]
penalty = cv_results_dataframe(search_logreg).param_base_model__logreg__penalty.head(1).values[0]
threshold = cv_results_dataframe(search_logreg).param_threshold.head(1).values[0]

print(f"Ответ: {precision}, {C}, {penalty}, {threshold}")

Ответ: 0.6, 1, l2, 0.46


### Задача 5. Используем дополнительные данные (3/3)
Сделайте то же самое для SVM. Запустите всё, как раньше, но на дополненных данных.

В поле ниже через запятую и пробел введите 3 значения: `precision` (округлите до 3 знаков), `C`, `threshold`.

Например, `0.543, 1000, 0.48`.

In [49]:
%%time

search_svc = GridSearchCV(
    svm_with_threshold, svm_grid,
    cv=splitter, scoring=scoring, refit=refit
)

search_svc.fit(X_train, y_train)
set_best_score(search_svc)

CPU times: user 1min 11s, sys: 52.1 s, total: 2min 3s
Wall time: 1min 12s


In [50]:
precision = round(cv_results_dataframe(search_svc).mean_test_precision.head(1).values[0], 3)
C = cv_results_dataframe(search_svc).param_base_model__svm__C.head(1).values[0]
threshold = cv_results_dataframe(search_svc).param_threshold.head(1).values[0]

print(f"Ответ: {precision}, {C}, {threshold}")

Ответ: 0.602, 0.1, 0.46


### Задача 6. Проверка на тестовой выборке (1/4)
#### Итоговая модель
Новые данные позволили немного улучшить результат, хоть и не очень сильно.

Метрики обеих моделей улучшились примерно одинаково. Формально, чуть выше метрика у SVM, но разница несущественная и, скорее всего, вызвана случайными факторами.

При этом логистическая регрессия — концептуально более простая модель, чем SVM. А если результаты одинаковы, зачем всё усложнять.

Итак, в качестве итоговой модели возьмём логистическую регрессию, обученную на дополненных данных.

Можете вытащить модель из `.best_estimator_` соответствующего grid search.

#### Предсказания
С помощью итоговой модели получите предсказания на тестовой выборке.

Получите отдельно как сами предсказания (0 или 1), так и предсказанные вероятности класса 1.

Проверим, что `threshold` работает как ожидается. Возьмите предсказанные вероятности. Вручную превратите их в предсказания, используя значение `threshold`, которое использует итоговая модель. Убедитесь, что результаты полностью совпали с тем, что модель выдаёт в качестве предсказанных классов. И убедитесь, что если использовать другое значение threshold, то результат уже не сойдётся.

#### Метрики на тесте
Посчитайте `precision` и `recall`. Что можно сказать об этих результатах?

В поле ниже введите через запятую и пробел полученные `precision` и `recall` (округлите до 3 знаков).

Например, `0.567, 0.765`.

In [51]:
best_model = search_logreg.best_estimator_
y_pred_proba_test = best_model.predict_proba(X_test)[:, 1]
y_pred_test = best_model.predict(X_test)
best_model.threshold

0.46

In [52]:
manual_predictions = (y_pred_proba_test >= best_model.threshold).astype(int)
print("Predictions match:", np.array_equal(y_pred_test, manual_predictions))

wrong_threshold_predictions = (y_pred_proba_test >= 0.5).astype(int)
print("Wrong threshold matches:", np.array_equal(y_pred_test, wrong_threshold_predictions))

Predictions match: True
Wrong threshold matches: False


In [53]:
from sklearn.metrics import precision_score, recall_score

precision_score(y_test, y_pred_test).round(3), recall_score(y_test, y_pred_test).round(3)

(0.605, 0.807)

### Задача 6. Проверка на тестовой выборке (2/4)
Хоть мы и оптимизируем `precision`, давайте в качестве практики посчитаем и другие метрики. И заодно построим пару кривых.

#### F1
1. На прошлом степе мы посчитали `precision` и `recall`. На основе этих значений прикиньте, каким примерно будет значение метрики F1 (или как минимум в каком интервале оно может находиться).
2. Чтобы лучше запомнить эту метрику, посчитайте её по формуле, не используя готовые функции.
3. Посчитайте F1 с помощью готовой функции из `sklearn.metrics`, сверьте ответ с предыдущим пунктом.

#### PR-кривая
Постройте PR-кривую и посмотрите на неё.

На графике мы можем видеть, что при `recall=1` кривая спускается к нашему бейслайну по `precision`. А уменьшение `recall'а` (движение влево по графику) позволяет увеличивать `precision`. В том числе видно полученное нами значение метрики при пороге 0.8. Ослабляя требование по `recall`, мы легко можем поднять `precision` до 0.7. И т.д.

#### PR-AUC
1. При построении PR-кривой у нас получились списки `precision'ов` и `recall'ов`. Используйте их чтобы посчитать PR-AUC.
2. Посчитайте PR-AUC другим способом: с помощью функции `average_precision_score` из `sklearn.metrics`. Т.к. эта функция использует немного другой способ расчёта, результат будет чуть отличаться (но не сильно).

#### ROC-кривая и ROC-AUC
Постройте ROC-кривую.

Посчитайте ROC-AUC через данные, полученные при построении ROC-кривой, а потом через готовую функцию. Убедитесь, что результаты полностью совпали.

Через запятую и пробел введите три значения: `F1`, `PR-AUC` (вариант через `precision_recall_curve` + `auc`) и `ROC-AUC`. Все значения округлите до 3 знаков. Например, `0.678, 0.678, 0.678`.

In [54]:
manual_f1 = 2 * (test_precision * test_recall) / (test_precision + test_recall)

sklearn_f1 = f1_score(y_test, y_pred_test)

print(f"Manual F1: {manual_f1:.3f}")
print(f"Sklearn F1: {sklearn_f1:.3f}")

NameError: name 'test_precision' is not defined

In [ ]:
y_pred_proba_test = best_model.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba_test)
pr_auc_manual = auc(recalls, precisions)
pr_auc_sklearn = average_precision_score(y_test, y_pred_proba_test)

print(f"PR-AUC (manual): {pr_auc_manual:.3f}")
print(f"PR-AUC (sklearn): {pr_auc_sklearn:.3f}")

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, roc_thresholds = roc_curve(y_test, y_pred_proba_test)
roc_auc_manual = auc(fpr, tpr)
roc_auc_sklearn = roc_auc_score(y_test, y_pred_proba_test)

print(f"ROC-AUC (manual): {roc_auc_manual:.3f}")
print(f"ROC-AUC (sklearn): {roc_auc_sklearn:.3f}")

In [ ]:
# PR-кривая
plt.figure(figsize=(15, 7))

plt.subplot(1, 2, 1)
plt.plot(recalls, precisions)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('PR-Curve')
plt.grid(True)

# ROC-кривая
plt.subplot(1, 2, 2)
plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-Curve')
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"Ответ: {sklearn_f1:.3f}, {pr_auc_manual:.3f}, {roc_auc_sklearn:.3f}")

### Задача 6. Проверка на тестовой выборке (3/4)
#### Коэффициенты модели
Помимо кривых и значений метрик интересно посмотреть, какие признаки оказались для модели важными.

Это можно делать разными способами. На следующем степе мы попробуем посчитать важности через `permutation importance`. А тут пока просто посмотрим на коэффициенты модели.

Как мы видели в одном из предыдущих уроков, для линейной регрессии коэффициенты непосредственно определяют прогноз модели: можно умножить их на значения признаков, добавить `intercept` и получить предсказание.

Логистическая регрессия работает чуть сложнее, т.к. это задача классификации. Но внутри у неё такие же коэффициенты, которые работают примерно по той же логике.

---
Достанем коэффициенты лучшей модели (замените на свои имена переменных и этапов пайплайна):

```python
best_model.base_model.named_steps['logreg'].coef_[0]
```

Названия столбцов тоже можно взять из самой модели, но из пайплайна предподготовки:

```python
best_model.base_model.named_steps['preprocessing'].get_feature_names_out()
```

Соберите из этого датафрейм, отсортируйте его по абсолютной величине коэффициента.

Для большей наглядности можно отрисовать в виде bar chart.

---
Посмотрите на результаты. Хотя коэффициенты модели не в полной мере отражают именно важность признаков, они дают какую-то информацию о ней. И помогают понять, как "рассуждает" модель. Большим плюсом этого подхода является и то, что у коэффициентов есть знаки. То есть мы не просто понимаем, что признак влияет на результат, но и в какую сторону он влияет.

Для категориальных признаков интерпретация немного усложняется. Из-за использования `drop=first` в `one-hot encoding`, например, нет коэффициента для `platform_android`. Этот вариант был принят моделью как "нейтральный", все остальные platform показывают смещение относительно него.

Но для числовых признаков всё прямолинейно.

Какие признаки связаны с уменьшением вероятности ухода клиента? То есть чем больше значение признака, тем менее вероятен отток и более вероятно, что клиент останется (с точки зрения полученной модели).

In [ ]:
coefficients = best_model.base_model.named_steps['logreg'].coef_[0]
feature_names = best_model.base_model.named_steps['preprocessing'].get_feature_names_out()

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
    'abs_coefficient': np.abs(coefficients)
})

coef_df_sorted = coef_df.sort_values('abs_coefficient', ascending=False)

print("Топ-15 наиболее важных признаков:")
coef_df_sorted.head(15)

In [ ]:
plt.figure(figsize=(12, 8))
top_features = coef_df_sorted.head(15)

plt.barh(top_features['feature'], top_features['coefficient'])
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.title('Logistic Regression Coefficients')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

Коэффициенты с минусом (которые на графике идут в левую сторону) уменьшают предсказание, т.е. двигают результат к классу 0. Чем больше значение признака и чем больше значение коэффициента, тем сильнее (но помните, что признак нормализованы). Коэффициенты с плюсом — наоборот двигают результат в сторону класса 1 (отток).

Для категориальных признаков интерпретация чуть сложнее из-за `drop='first'`. Например, мы видим, что все три `user_type_cars_name` отложены вправо. Тут немного не повезло: `drop='first'` выкинул тип `cars_dealer`, который представляет профессионалов и, конечно, является самым "анти-отточным". Он представляет собой "нейтральный уровень" и остальные категории как бы считаются относительно него. Если бы `drop=first` выкинул `simple` или `unknown`, мы бы увидели большие столбики влево для `dealer` и `seller`.

In [ ]:
negative_coef = coef_df[coef_df['coefficient'] < 0].sort_values('coefficient')

print("\nПризнаки, снижающие вероятность ухода (отрицательные коэффициенты):")
negative_coef[['feature', 'coefficient']].head(4)

### Задача 6. Проверка на тестовой выборке (4/4)
#### Permutation importance
Теперь посчитаем `permutation importance`. Как сильно ухудшится значение метрики, если взять и перемешать значение какого-нибудь столбца. Так чтобы сам столбец остался, но перестал содержать осмысленную информацию.

Тут встаёт вопрос, какую метрику использовать: что именно должно ухудшаться или не ухудшаться, когда мы портим столбец. Если не указать ничего, будет accuracy. В таком варианте мы тоже сможем получить какую-то информацию о важности признаков, но это не та метрика, которую мы оптимизируем.

Если поставить `precision`, то мы будем игнорировать, как меняется `recall`, когда будет "портиться" признак. Это тоже не совсем то, что мы хотим: если, например, `precision` не изменится, но сильно упадёт `recall` — для нас это важно, а permutation_importance этого не поймает.

Если написать кастомную метрику "`precision`, если `recall > 0.8`, иначе 0", то она будет зануляться даже если `recall` вместо 0.8 станет 0.799. Это тоже испортит нам всю картину. Что делать?

Вспомним, про PR-кривую. Там можно посмотреть, какое значение `precision` получается при `recall = 0.8`.

Это не один в один наша метрика, но по смыслу она очень близка (зафиксирован `recall`, смотрим `precision` при таком ограничении). И её довольно просто посчитать.

Вот реализация самой метрики:

```python
def precision_at_recall(y_true, y_pred_proba, target_recall=0.8):
    """Find precision at target recall using PR curve."""
    precisions, recalls, _ = precision_recall_curve(y_true, y_pred_proba)
    valid_idx = recalls >= target_recall
    return precisions[valid_idx][-1]
```

Получаем данные для PR-кривой. Оставляем только такие `precision`, где `recall` не меньше 0.8. Берём из них последнюю точку, с самым низким `recall` (т.е. такую, где `recall` максимально близок к 0.8; `recalls` идут в порядке убывания, поэтому берём `[-1]`, а не `[0]`).

Вызвовите эту метрику на наших данных, чтобы посмотреть, как она работает.

А в `permutation_importance` передавайте её как `scoring=make_scorer(precision_at_recall, needs_proba=True)` (параметр `needs_proba` нужен чтобы функции передавались именно вероятности, а не классы).

Посчитайте `permutation_importance`, используя параметры `n_repeats=10`, `random_state=42` и `scoring`, который мы обсудили выше. Визуализируйте результаты с помощью горизонтального `bar chart`.

Что можно сказать о результатах?

In [ ]:
def precision_at_recall(y_true, y_pred_proba, target_recall=0.8):
    """Find precision at target recall using PR curve."""
    precisions, recalls, _ = precision_recall_curve(y_true, y_pred_proba)
    valid_idx = recalls >= target_recall
    return precisions[valid_idx][-1]

In [ ]:
y_pred_proba_test = best_model.predict_proba(X_test)[:, 1]
precision_at_08 = precision_at_recall(y_test, y_pred_proba_test, target_recall=0.8)
print(f"Precision при recall=0.8: {precision_at_08:.3f}")

In [ ]:
custom_scorer = make_scorer(
    precision_at_recall, 
    needs_proba=True,
    greater_is_better=True
)

In [ ]:
perm_importance = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring=custom_scorer,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

In [ ]:
feature_names

In [ ]:
perm_importance.importances_std

In [ ]:
perm_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=False)

perm_df

In [ ]:
plt.figure(figsize=(12, 8))
top_perm = perm_df.head(15)

plt.barh(top_perm['feature'], top_perm['importance_mean'], 
         xerr=top_perm['importance_std'], capsize=5)
plt.xlabel('Уменьшение precision при recall=0.8')
plt.title('Permutation Importance (precision при recall=0.8)')
plt.gca().invert_yaxis()
plt.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
comparison_df = pd.merge(coef_df, perm_df, on='feature')
comparison_df = comparison_df.sort_values('importance_mean', ascending=False)

print("\nСравнение коэффициентов и permutation importance:")
comparison_df[['feature', 'coefficient', 'importance_mean']].head(10)